# Monet Style Transfer

CycleGAN implementation and Stable Diffusion inference with an existing Monet LoRA.
See the README for the original saved visual results, data setup and limitations.
Training and bulk export are disabled by default. No model was rerun while preparing this notebook.


## 1. CycleGAN


In [ ]:
import numpy as np
import os
from pathlib import Path

# Run Jupyter from the repository root; update these paths for Kaggle if needed.
BASE_INPUT = Path("data/gan-getting-started")
OUTPUT_ROOT = Path("outputs")
WEIGHTS_PATH = Path("models/cyclegan_weights.pth")
TRAIN_FROM_SCRATCH = False
EXPORT_ALL_IMAGES = False
NUM_WORKERS = 0 if os.name == "nt" else 2


In [ ]:
import glob
import torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader

### Data and augmentation


In [ ]:
class ImageTransform:
    def __init__(self, load_size=286, crop_size=256):
        # Training: resize to 286 and crop to 256 with random augmentation
        self.train_transform = T.Compose([
            T.Resize((load_size, load_size), antialias=True),
            T.RandomCrop(crop_size),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),  # map to [-1, 1] to match the generator's Tanh output
        ])
        # Validation/test: resize without augmentation
        self.val_transform = T.Compose([
            T.Resize((crop_size, crop_size), antialias=True),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def __call__(self, img, train=True):
        return self.train_transform(img) if train else self.val_transform(img)


In [ ]:
class MonetDataset(Dataset):
    def __init__(self, file_paths, transform, train=True):
        self.files     = file_paths
        self.transform = transform
        self.train     = train

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        return self.transform(img, self.train)


In [ ]:
MONET_PATH = os.path.join(BASE_INPUT, "monet_jpg/*.jpg")
PHOTO_PATH = os.path.join(BASE_INPUT, "photo_jpg/*.jpg")

monet_files = sorted(glob.glob(MONET_PATH))
photo_files = sorted(glob.glob(PHOTO_PATH))

assert len(photo_files) > 0, f"No photographs found in {PHOTO_PATH}"
assert len(monet_files) > 0, f"No Monet images found in {MONET_PATH}"

transform = ImageTransform(load_size=286, crop_size=256)

# Split photographs 75/25; use all Monet images for training
n_train = int(0.75 * len(photo_files))
train_monet = MonetDataset(monet_files, transform, train=True)
train_photo = MonetDataset(photo_files[:n_train], transform, train=True)
val_photo   = MonetDataset(photo_files[n_train:], transform, train=False)

loader_cfg = dict(batch_size=4, num_workers=NUM_WORKERS, pin_memory=True)
monet_loader = DataLoader(train_monet, shuffle=True, drop_last=True, **loader_cfg)
photo_loader = DataLoader(train_photo, shuffle=True, drop_last=True, **loader_cfg)
val_loader   = DataLoader(val_photo,   shuffle=False, **loader_cfg)

print(f"Monets : {len(monet_files)} | Photos train : {len(train_photo)} | val : {len(val_photo)} | steps/epoch : {len(photo_loader)}")


### Hyperparameters and device


In [ ]:
import time, zipfile, itertools, os
from tqdm import tqdm
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

N_EPOCHS = 10
LAMBDA_CYCLE = 10.0 # cycle-consistency weight
LAMBDA_ID = 5.0 # identity-loss weight
LR = 2e-4
BETA1 = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', device)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = True  # optimize CUDA kernels for fixed input sizes


### CycleGAN architecture


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class CycleGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_features=64, n_residual=6):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, n_features, 7),
            nn.InstanceNorm2d(n_features),
            nn.ReLU(True),
            nn.Conv2d(n_features, n_features * 2, 3, stride=2, padding=1),
            nn.InstanceNorm2d(n_features * 2),
            nn.ReLU(True),
            nn.Conv2d(n_features * 2, n_features * 4, 3, stride=2, padding=1),
            nn.InstanceNorm2d(n_features * 4),
            nn.ReLU(True),
        )
        self.residuals = nn.Sequential(
            *[ResidualBlock(n_features * 4) for _ in range(n_residual)]
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(n_features * 4, n_features * 2, 3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(n_features * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(n_features * 2, n_features, 3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(n_features),
            nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(n_features, out_channels, 7),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.decoder(self.residuals(self.encoder(x)))


_g = CycleGenerator(n_residual=6).to(device)
_x = torch.randn(1, 3, 256, 256, device=device)
print(f'Generator : {sum(p.numel() for p in _g.parameters())/1e6:.1f}M params')
print(f'  {_x.shape} -> {_g(_x).shape}')
del _g, _x


In [ ]:
class PatchGANDiscriminator(nn.Module):
    """70x70 PatchGAN: classify patches rather than the whole image."""
    def __init__(self, in_channels=3, n_features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, n_features, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(n_features, n_features * 2, 4, stride=2, padding=1),
            nn.InstanceNorm2d(n_features * 2),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(n_features * 2, n_features * 4, 4, stride=2, padding=1),
            nn.InstanceNorm2d(n_features * 4),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(n_features * 4, n_features * 8, 4, stride=1, padding=1),
            nn.InstanceNorm2d(n_features * 8),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(n_features * 8, 1, 4, stride=1, padding=1),
        )

    def forward(self, x):
        return self.model(x)


_d = PatchGANDiscriminator().to(device)
_x = torch.randn(1, 3, 256, 256, device=device)
print(f'Discriminator : {sum(p.numel() for p in _d.parameters())/1e3:.1f}K params')
print(f'  {_x.shape} -> patch map {_d(_x).shape}')
del _d, _x


In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'InstanceNorm' in classname and m.weight is not None:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


G_p2m = CycleGenerator(n_residual=6).to(device)
G_m2p = CycleGenerator(n_residual=6).to(device)
D_monet = PatchGANDiscriminator().to(device)
D_photo = PatchGANDiscriminator().to(device)

for net in [G_p2m, G_m2p, D_monet, D_photo]:
    net.apply(weights_init)

print(f'G_p2m : {sum(p.numel() for p in G_p2m.parameters())/1e6:.1f}M params')
print(f'G_m2p : {sum(p.numel() for p in G_m2p.parameters())/1e6:.1f}M params')


### Losses and optimizers


In [ ]:
criterion_GAN = nn.MSELoss() #LSGAN
criterion_cycle = nn.L1Loss()
criterion_id = nn.L1Loss()

def denorm(t):
    # undo normalization for display and saving: [-1, 1] to [0, 1]
    return (t * 0.5 + 0.5).clamp(0, 1)

# Optimizer for both generators
optimizer_G = torch.optim.Adam(
    itertools.chain(G_p2m.parameters(), G_m2p.parameters()),
    lr=LR, betas=(BETA1, 0.999)
)
optimizer_Dm = torch.optim.Adam(D_monet.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_Dp = torch.optim.Adam(D_photo.parameters(), lr=LR, betas=(BETA1, 0.999))

# Automatic mixed precision reduces GPU memory use
use_amp = device.type == 'cuda'
scaler_G = torch.amp.GradScaler('cuda', enabled=use_amp)
scaler_Dm = torch.amp.GradScaler('cuda', enabled=use_amp)
scaler_Dp = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f'AMP enabled : {use_amp}')


### Load a checkpoint or explicitly enable training


In [ ]:
history = {'G': [], 'D_monet': [], 'D_photo': [], 'cycle': [], 'identity': [],
           'd_real_m': [], 'd_fake_m': [], 'd_real_p': [], 'd_fake_p': []}

if WEIGHTS_PATH.is_file():
    checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
    G_p2m.load_state_dict(checkpoint['G_p2m'])
    G_m2p.load_state_dict(checkpoint['G_m2p'])
    print(f"Weights loaded (epoch {checkpoint['epoch']})")

elif TRAIN_FROM_SCRATCH:
    print("Training from scratch...")

    def infinite(loader):
        while True:
            yield from loader

    monet_iter = infinite(monet_loader)
    steps_per_epoch = len(photo_loader)
    start = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        G_p2m.train(); G_m2p.train(); D_monet.train(); D_photo.train()

        sum_G = sum_Dm = sum_Dp = sum_cyc = sum_id = 0.0
        sum_dr_m = sum_df_m = sum_dr_p = sum_df_p = 0.0

        for photo in tqdm(photo_loader, desc=f'Epoch {epoch:02d}/{N_EPOCHS}', leave=False):
            photo = photo.to(device, non_blocking=True)
            monet = next(monet_iter).to(device, non_blocking=True)

            # Generators
            optimizer_G.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                fake_monet = G_p2m(photo)
                fake_photo = G_m2p(monet)

                # Adversarial: G aims to fool D (target = 1)
                pred_fm = D_monet(fake_monet)
                pred_fp = D_photo(fake_photo)
                loss_adv = (criterion_GAN(pred_fm, torch.ones_like(pred_fm))
                          + criterion_GAN(pred_fp, torch.ones_like(pred_fp)))

                # Cycle consistency : photo -> monet -> photo ≈ photo
                loss_cyc = (criterion_cycle(G_m2p(fake_monet), photo)
                          + criterion_cycle(G_p2m(fake_photo), monet)) * LAMBDA_CYCLE

                # Identity : G_p2m(monet) ≈ monet (preserve colors)
                loss_id = (criterion_id(G_p2m(monet), monet)
                         + criterion_id(G_m2p(photo), photo)) * LAMBDA_ID

                loss_G = loss_adv + loss_cyc + loss_id

            scaler_G.scale(loss_G).backward()
            scaler_G.step(optimizer_G)
            scaler_G.update()

            # Discriminator Monet
            optimizer_Dm.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                pred_real_m = D_monet(monet)
                pred_fake_m = D_monet(fake_monet.detach())  # detach : do not backpropagate to G
                loss_Dm = (criterion_GAN(pred_real_m, torch.ones_like(pred_real_m))
                         + criterion_GAN(pred_fake_m, torch.zeros_like(pred_fake_m))) * 0.5

            scaler_Dm.scale(loss_Dm).backward()
            scaler_Dm.step(optimizer_Dm)
            scaler_Dm.update()

            # Discriminator Photo
            optimizer_Dp.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                pred_real_p = D_photo(photo)
                pred_fake_p = D_photo(fake_photo.detach())
                loss_Dp = (criterion_GAN(pred_real_p, torch.ones_like(pred_real_p))
                         + criterion_GAN(pred_fake_p, torch.zeros_like(pred_fake_p))) * 0.5

            scaler_Dp.scale(loss_Dp).backward()
            scaler_Dp.step(optimizer_Dp)
            scaler_Dp.update()

            # Accumulate metrics
            sum_G   += loss_G.item();  sum_Dm += loss_Dm.item(); sum_Dp += loss_Dp.item()
            sum_cyc += loss_cyc.item(); sum_id += loss_id.item()
            sum_dr_m += torch.sigmoid(pred_real_m).mean().item()
            sum_df_m += torch.sigmoid(pred_fake_m).mean().item()
            sum_dr_p += torch.sigmoid(pred_real_p).mean().item()
            sum_df_p += torch.sigmoid(pred_fake_p).mean().item()

        # End of epoch
        n = steps_per_epoch
        history['G'].append(sum_G / n)
        history['D_monet'].append(sum_Dm / n)
        history['D_photo'].append(sum_Dp / n)
        history['cycle'].append(sum_cyc / n)
        history['identity'].append(sum_id / n)
        history['d_real_m'].append(sum_dr_m / n); history['d_fake_m'].append(sum_df_m / n)
        history['d_real_p'].append(sum_dr_p / n); history['d_fake_p'].append(sum_df_p / n)

        elapsed = time.time() - start
        restant = elapsed / epoch * (N_EPOCHS - epoch)
        print(f'Epoch {epoch:02d}/{N_EPOCHS} | G={sum_G/n:.3f} D_m={sum_Dm/n:.3f} D_p={sum_Dp/n:.3f} | '
              f'cyc={sum_cyc/n:.3f} id={sum_id/n:.3f} | {elapsed/60:.1f}min (~{restant/60:.0f}min remaining)')

    print('Training complete.')

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    torch.save({"G_p2m": G_p2m.state_dict(), "G_m2p": G_m2p.state_dict(),
                "epoch": N_EPOCHS}, OUTPUT_ROOT / "cyclegan_weights.pth")
else:
    raise FileNotFoundError(
        f"Checkpoint not found: {WEIGHTS_PATH}. Set TRAIN_FROM_SCRATCH=True to train, "
        "or skip to the independent Stable Diffusion section."
    )


### Visualize results


In [ ]:
if history['G']:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # Adversarial losses
    axes[0].plot(history['G'],       label='G total', color='blue')
    axes[0].plot(history['D_monet'], label='D Monet',  color='red')
    axes[0].plot(history['D_photo'], label='D Photo',  color='orange')
    axes[0].set_xlabel('Epoque'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Adversarial losses (LSGAN)'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # Discriminator scores
    axes[1].plot(history['d_real_m'], label='D_monet(vrai)',  color='green')
    axes[1].plot(history['d_fake_m'], label='D_monet(faux)',  color='limegreen', linestyle='--')
    axes[1].plot(history['d_real_p'], label='D_photo(vraie)', color='steelblue')
    axes[1].plot(history['d_fake_p'], label='D_photo(fausse)', color='skyblue', linestyle='--')
    axes[1].axhline(y=0.5, color='gray', linestyle=':', alpha=0.7, label='Equilibre')
    axes[1].set_xlabel('Epoque'); axes[1].set_ylabel('Score moyen')
    axes[1].set_title('Discriminator scores'); axes[1].legend(fontsize=8)
    axes[1].set_ylim(0, 1); axes[1].grid(alpha=0.3)

    # Raw cycle and identity losses (without weights)
    axes[2].plot([v / LAMBDA_CYCLE for v in history['cycle']],    label='Cycle (L1)', color='green')
    axes[2].plot([v / LAMBDA_ID   for v in history['identity']], label='Identity (L1)', color='purple')
    axes[2].set_xlabel('Epoque'); axes[2].set_ylabel('Loss L1')
    axes[2].set_title('Pertes cycle & identity (sans lambda)'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle('Diagnostic CycleGAN', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Training curves unavailable when loading weights from a file.")


In [ ]:
from torchvision.utils import make_grid

G_p2m.eval(); G_m2p.eval()

val_photos = []
for img in val_loader:
    val_photos.append(img)
    if len(val_photos) >= 6:
        break

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
row_titles = ['Photo originale', 'Fake Monet (G_p2m)', 'Photo reconstruite (cycle)']

with torch.no_grad():
    for col, photo in enumerate(val_photos[:6]):
        photo = photo.to(device)
        fake_m = G_p2m(photo)
        rec_p  = G_m2p(fake_m)
        for row, img in enumerate([photo, fake_m, rec_p]):
            ax = axes[row, col]
            npimg = denorm(img[0]).cpu().permute(1, 2, 0).numpy()
            ax.imshow(npimg); ax.axis('off')
            if col == 0:
                ax.set_ylabel(row_titles[row], fontsize=9, rotation=0,
                              labelpad=120, va='center')

plt.suptitle('Cycle consistency : photo -> Monet -> photo reconstruite', fontsize=13)
plt.tight_layout(); plt.show()


### Optional image export


In [ ]:
if EXPORT_ALL_IMAGES:
    from PIL import Image as PILImage
    
    OUTPUT_DIR = str(OUTPUT_ROOT / 'images')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Generate all photographs, including those outside the training partition
    all_dataset = MonetDataset(photo_files, transform, train=False)
    all_loader  = DataLoader(all_dataset, batch_size=4, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    G_p2m.eval()
    n_total = len(photo_files)  # original competition photograph count: 7028
    idx = 1
    with torch.no_grad():
        for batch in tqdm(all_loader, desc='Generation Monet'):
            batch = batch.to(device, non_blocking=True)
            for img_t in denorm(G_p2m(batch)):
                if idx > n_total:  # do not exceed the input image count
                    break
                arr = (img_t.cpu().permute(1, 2, 0).numpy() * 255).astype('uint8')
                PILImage.fromarray(arr).resize((256, 256), PILImage.LANCZOS).save(
                    os.path.join(OUTPUT_DIR, f'{idx}.jpg'), quality=95
                )
                idx += 1
    
    print(f'{idx - 1} images saved in {OUTPUT_DIR}')
    
    ZIP_PATH = str(OUTPUT_ROOT / 'images.zip')
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(OUTPUT_DIR), key=lambda x: int(x.split('.')[0])):
            zf.write(os.path.join(OUTPUT_DIR, fname), arcname=fname)
    
    print(f'ZIP saved : {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} Mo)')
else:
    print("Bulk export disabled. Set EXPORT_ALL_IMAGES=True to enable it.")


## 2. Stable Diffusion with an existing Monet LoRA


Use pretrained Stable Diffusion with the existing J-YOON Monet LoRA. This section performs inference only; it does not train an adapter.


In [ ]:
import os, glob, zipfile
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from diffusers import StableDiffusionImg2ImgPipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', device)

# Reuse BASE_INPUT from the configuration cell.
photo_files = sorted(glob.glob(os.path.join(BASE_INPUT, 'photo_jpg/*.jpg')))
print(f'Photos : {len(photo_files)}')

### Load the pretrained pipeline and adapter


In [ ]:
# Load the pipeline in float16
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    safety_checker=None,
).to(device)
pipe.enable_attention_slicing()

# Existing Monet LoRA from Hugging Face (J-YOON/lora-monet-sd1.5)
# Trigger word : "msl monet" (include in the prompt to activate the style)
pipe.load_lora_weights(
    'J-YOON/lora-monet-sd1.5',
    weight_name='pytorch_lora_weights.safetensors',
)

print('Stable Diffusion and Monet LoRA loaded.')


### Preview a few photographs


In [ ]:
PROMPT   = 'msl monet'
STRENGTH = 0.20   # 0=original photograph, 1=full generation
STEPS    = 100

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

with torch.no_grad():
    for col, path in enumerate(photo_files[:6]):
        orig   = Image.open(path).convert('RGB').resize((512, 512))
        result = pipe(
            prompt=PROMPT, image=orig,
            strength=STRENGTH, guidance_scale=7.0,
            num_inference_steps=STEPS,
        ).images[0]

        axes[0, col].imshow(orig);    axes[0, col].axis('off')
        axes[1, col].imshow(result);  axes[1, col].axis('off')
        if col == 0:
            axes[0, col].set_ylabel('Photo', fontsize=9)
            axes[1, col].set_ylabel('Style Monet (SD+LoRA)', fontsize=9)

plt.suptitle(f'Stable Diffusion img2img + LoRA Monet  (strength={STRENGTH})', fontsize=12)
plt.tight_layout(); plt.show()
